# Prepare Dataset v2: Only Complete Conversations
Training data = complete conversations only (all end with `<|im_end|>`)
Test data = positive (complete) + negative (truncated) for evaluation

In [59]:
import os
import json
import random
import numpy as np
from glob import glob
from tqdm import tqdm
from datasets import load_dataset
from collections import Counter, defaultdict
from transformers import AutoTokenizer
from chinidataset import ParquetWriter, StreamingDataset
from multiprocess import Pool
import itertools
import subprocess

random.seed(42)
np.random.seed(42)

## Config

In [60]:
BLOCK_SIZE = 8192
N_WORKERS = 20
TOKENIZER_NAME = 'Qwen/Qwen3-0.6B'
OUTPUT_DIR = './parquet-train-v2'
MERGED_DIR = './parquet-train-merged-v2'
TEST_DIR = './parquet-test-v2'
S3_BUCKET = 'aies-research-datasets'
S3_PREFIX = 'call-center-language-switching-v2'

## Converters

In [61]:
def convert_messages_format(row):
    if 'messages' in row:
        return row['messages']
    return None


def convert_sharegpt_format(row):
    role_map = {
        'system': 'system', 'human': 'user', 'gpt': 'assistant',
        'user': 'user', 'assistant': 'assistant',
    }
    if 'conversations' in row:
        messages = []
        for turn in row['conversations']:
            role = role_map.get(turn.get('from', turn.get('role', '')), 'user')
            content = turn.get('value', turn.get('content', ''))
            messages.append({'role': role, 'content': content})
        return messages
    return None


def _extract_text_content(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict):
                if part.get('type') == 'text':
                    parts.append(part.get('text', ''))
                elif part.get('type') == 'audio':
                    continue
            elif isinstance(part, str):
                parts.append(part)
        return ' '.join(parts) if parts else None
    return None


def convert_json_string_format(row):
    raw = row.get('conversation') or row.get('prompt')
    if raw is None or not isinstance(raw, str):
        return None
    try:
        parsed = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(parsed, list) or len(parsed) == 0:
        return None
    messages = []
    system_prompt = row.get('system_prompt')
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    for turn in parsed:
        role = turn.get('role', 'user')
        content = _extract_text_content(turn.get('content', ''))
        if content:
            messages.append({'role': role, 'content': content})
    return messages if len(messages) >= 2 else None


def convert_function_call_format(row):
    if 'examples' not in row or 'function' not in row:
        return None
    examples = row['examples']
    if not isinstance(examples, list) or len(examples) == 0:
        return None
    func_spec = row['function']
    if isinstance(func_spec, str):
        func_desc = func_spec
    elif isinstance(func_spec, dict):
        func_desc = json.dumps(func_spec)
    else:
        func_desc = str(func_spec)
    messages = [{'role': 'system', 'content': f'You have access to the following function:\n{func_desc}'}]
    for ex in examples:
        if isinstance(ex, dict):
            if 'user' in ex:
                messages.append({'role': 'user', 'content': ex['user']})
            if 'assistant' in ex:
                messages.append({'role': 'assistant', 'content': ex['assistant']})
    return messages if len(messages) >= 3 else None


def get_language(row, source_name, split_name=None):
    """Extract language from row based on dataset source."""
    # Call-Center: has source_language + switch_language
    if 'source_language' in row:
        src = row['source_language']
        sw = row.get('switch_language', '')
        return f'{src}-{sw}' if sw and sw != src else src
    # Multiturn-Chat: split name is the language (malay, mixed_manglish)
    if split_name:
        return split_name
    # Speech-Instructions: dataset field
    if 'dataset' in row and source_name == 'speech-instructions':
        return 'malay'
    # Function-Call: domain field, multilingual
    if 'domain' in row:
        return 'multilingual'
    # Fallback
    return row.get('language', row.get('lang', 'unknown'))


def convert_to_unified(row, source_name, split_name=None):
    messages = convert_messages_format(row)
    if messages is None:
        messages = convert_sharegpt_format(row)
    if messages is None:
        messages = convert_json_string_format(row)
    if messages is None:
        messages = convert_function_call_format(row)
    if messages is None:
        return None
    return {
        'messages': messages,
        'language': get_language(row, source_name, split_name),
        'source': source_name,
    }

## Chat template builders

In [62]:
def format_chat_template(messages):
    parts = []
    for msg in messages:
        parts.append(f'<|im_start|>{msg["role"]}\n{msg["content"]}<|im_end|>')
    return '\n'.join(parts)


def build_positive(messages):
    """Complete conversation ending with <|im_end|>."""
    return format_chat_template(messages)


def build_negative(messages):
    """Truncated conversation NOT ending with <|im_end|>.
    Always truncates the last message content to make it clearly incomplete.
    """
    text = format_chat_template(messages)
    # Strip trailing <|im_end|>
    if text.endswith('<|im_end|>'):
        text = text[:-len('<|im_end|>')]

    # Find the last <|im_start|> block (any role) and truncate within it
    last_im_start = text.rfind('<|im_start|>')
    if last_im_start >= 0:
        # Find the content start (after "role\n")
        newline_after_role = text.find('\n', last_im_start)
        if newline_after_role >= 0:
            content_start = newline_after_role + 1
            content = text[content_start:]
            if len(content) > 10:
                keep_ratio = random.uniform(0.2, 0.7)
                keep_len = max(5, int(len(content) * keep_ratio))
                text = text[:content_start + keep_len]

    return text.rstrip()

## Multipacking helpers

In [63]:
COLUMNS = {
    'input_ids': 'uint32[]',
    'position_ids': 'uint32[]',
    'attention_mask': 'uint32[]',
    'text': 'str',
}


def collator_pack(batch_ids, batch_position_ids):
    input_ids, position_ids, masks = [], [], []
    for i in range(len(batch_ids)):
        input_ids.extend(batch_ids[i])
        position_ids.extend(batch_position_ids[i])
        masks.append(len(batch_ids[i]))
    return {
        'input_ids': np.array(input_ids, dtype=np.uint32),
        'position_ids': np.array(position_ids, dtype=np.uint32),
        'attention_mask': np.array(masks, dtype=np.uint32),
        'text': '',
    }


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield (lst[i:i + n], i // n)


def multiprocessing_run(data, function, cores=6, returned=True):
    df_split = chunks(data, max(1, len(data) // cores))
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()
    if returned:
        return list(itertools.chain(*pooled))


tokenizer = None


def write_parquet_worker(args):
    rows, index = args
    out_root = f'{OUTPUT_DIR}/shard-{index}'

    count, written = 0, 0
    temp_ids, temp_pos = [], []

    with ParquetWriter(out=out_root, columns=COLUMNS, exist_ok=True) as writer:
        for row in tqdm(rows, desc=f'Worker {index}'):
            outputs = tokenizer(row['text'], add_special_tokens=False)
            ids = outputs['input_ids']
            length = len(ids)

            if length == 0 or length > BLOCK_SIZE:
                continue

            if count + length > BLOCK_SIZE:
                o = collator_pack(temp_ids, temp_pos)
                if o['input_ids'].shape[0] > 0:
                    writer.write(o)
                    written += 1
                temp_ids = [ids]
                temp_pos = [list(range(length))]
                count = length
            else:
                temp_ids.append(ids)
                temp_pos.append(list(range(length)))
                count += length

        if temp_ids:
            o = collator_pack(temp_ids, temp_pos)
            if o['input_ids'].shape[0] > 0:
                writer.write(o)
                written += 1

    return [written]

## Step 1: Download datasets

In [ ]:
ds_call_center = load_dataset('Scicom-intl/Call-Center-Language-Switching')
ds_multiturn = load_dataset('mesolitica/Malaysian-Multiturn-Chat-Assistant')
ds_speech = load_dataset('mesolitica/Malaysian-Speech-Instructions')

# Drop audio columns
for split in ds_speech:
    audio_cols = [c for c in ds_speech[split].column_names
                  if type(ds_speech[split].features[c]).__name__ == 'Audio']
    if audio_cols:
        ds_speech[split] = ds_speech[split].remove_columns(audio_cols)

FUNCTION_CALL_CONFIGS = [
    'extended_functions', 'extended_functions_v2',
    'functions', 'functions_multilingual_examples',
    'functions_multilingual_examples_v2',
    'multifunctions', 'multifunctions_deep', 'multifunctions_deep_v2',
    'multifunctions_multiturn', 'multifunctions_multiturn_extra',
    'multifunctions_multiturn_language_switching',
    'multifunctions_multiturn_language_switching_extra',
    'multifunctions_multiturn_v2_deep',
    'multifunctions_v2',
    'telco_multifunctions_premium', 'telco_multifunctions_premium_multiturn',
]
ds_function_call_all = {}
for cfg in FUNCTION_CALL_CONFIGS:
    print(f'  Loading Function-Call/{cfg}...')
    ds_function_call_all[cfg] = load_dataset('Scicom-intl/Function-Call', cfg)

## Step 2: Convert to unified format

In [ ]:
all_data = []

for name, ds in [
    ('call-center-language-switching', ds_call_center),
    ('multiturn-chat', ds_multiturn),
    ('speech-instructions', ds_speech),
]:
    print(f'\nConverting {name}...')
    for split in ds:
        print(f'  split: {split}')
        for row in tqdm(ds[split]):
            unified = convert_to_unified(row, name, split_name=split)
            if unified:
                all_data.append(unified)

for cfg, ds in ds_function_call_all.items():
    print(f'\nConverting function-call/{cfg}...')
    split = 'train'
    if split not in ds:
        split = list(ds.keys())[0]
    for row in tqdm(ds[split]):
        unified = convert_to_unified(row, f'function-call/{cfg}')
        if unified:
            all_data.append(unified)

print(f'\nTotal unified conversations: {len(all_data)}')
source_counts = Counter(d['source'] for d in all_data)
for src, cnt in source_counts.items():
    print(f'  {src}: {cnt}')

# Show language distribution
lang_counts = Counter(d['language'] for d in all_data)
print(f'\nLanguages:')
for lang, cnt in lang_counts.most_common():
    print(f'  {lang}: {cnt}')

## Step 3: Build test set

In [ ]:
call_center_data = [d for d in all_data if d['source'] == 'call-center-language-switching']
by_language = defaultdict(list)
for d in call_center_data:
    by_language[d['language']].append(d)

print('Languages found:')
for lang, items in by_language.items():
    print(f'  {lang}: {len(items)} conversations')

test_set = []
test_indices = set()
for lang, items in by_language.items():
    n_sample = min(50, len(items))
    sampled = random.sample(items, n_sample)
    for s in sampled:
        s['split'] = 'test'
        test_set.append(s)
        test_indices.add(id(s))
    print(f'  Test: {lang} -> {n_sample} conversations')

train_data = [d for d in all_data if id(d) not in test_indices]
print(f'\nTotal test set:  {len(test_set)} conversations')
print(f'Total train set: {len(train_data)} conversations')

## Step 4: Build samples
**Training = only complete conversations (positive)**

**Test = positive + negative for evaluation**

In [68]:
# TRAINING: only complete conversations
train_samples = []
for d in tqdm(train_data, desc='Building train samples'):
    messages = d['messages']
    if not messages or len(messages) < 2:
        continue
    train_samples.append({'text': build_positive(messages), 'label': 'positive', 'source': d['source']})
random.shuffle(train_samples)

# TEST: both positive and negative
test_samples = []
for d in tqdm(test_set, desc='Building test samples'):
    messages = d['messages']
    if not messages or len(messages) < 2:
        continue
    test_samples.append({'text': build_positive(messages), 'label': 'positive', 'language': d['language']})
    test_samples.append({'text': build_negative(messages), 'label': 'negative', 'language': d['language']})

print(f'\nTrain samples: {len(train_samples)} (positive only)')
print(f'Test  samples: {len(test_samples)} (positive + negative)')

Building test samples: 100%|██████████| 600/600 [00:00<00:00, 57095.14it/s]


Train samples: 430133 (positive only)
Test  samples: 1200 (positive + negative)


## Step 5: Tokenize & multipack

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
print(f'Vocab size: {len(tokenizer)}')
print(f'<|im_end|> token ID: {im_end_id}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
results = multiprocessing_run(train_samples, write_parquet_worker, cores=N_WORKERS)
print(f'Total blocks written across {N_WORKERS} workers: {sum(results)}')

Vocab size: 151669
<|im_end|> token ID: 151645


Directory /root/small-ablation/call-center-language-switching/parquet-train-v2/shard-0 exists; removing contents.
Worker 14:  29%|██▉       | 6287/21506 [00:25<00:53, 281.85it/s]

## Step 6: Merge shards

In [38]:
shard_folders = sorted(
    glob(f'{OUTPUT_DIR}/shard-*'),
    key=lambda x: int(x.split('-')[-1]),
)
print(f'Found {len(shard_folders)} shards')

total = 0
with ParquetWriter(out=MERGED_DIR, columns=COLUMNS, exist_ok=True) as writer:
    for folder in shard_folders:
        try:
            ds = StreamingDataset(local=folder)
            for i in tqdm(range(len(ds)), desc=folder):
                writer.write(ds[i])
                total += 1
        except Exception as e:
            print(f'Error reading {folder}: {e}')
print(f'Merged dataset: {total} blocks -> {MERGED_DIR}')

Directory /root/small-ablation/call-center-language-switching/parquet-train-merged-v2 exists; removing contents.


Found 21 shards


./parquet-train-v2/shard-20: 100%|██████████| 3/3 [00:00<00:00, 826.46it/s]

Merged dataset: 95488 blocks -> ./parquet-train-merged-v2


## Step 7: Write test parquet

In [70]:
total_test = 0
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
with ParquetWriter(out=TEST_DIR, columns=COLUMNS, exist_ok=True) as writer:
    for sample in tqdm(test_samples, desc='Writing test parquet'):
        outputs = tokenizer(sample['text'], add_special_tokens=False)
        ids = outputs['input_ids']
        if len(ids) == 0:
            continue
        writer.write({
            'input_ids': np.array(ids, dtype=np.uint32),
            'position_ids': np.array(list(range(len(ids))), dtype=np.uint32),
            'attention_mask': np.array([len(ids)], dtype=np.uint32),
            'text': json.dumps({'label': sample['label'], 'language': sample['language']}),
        })
        total_test += 1
print(f'Test set: {total_test} samples -> {TEST_DIR}')

Directory /root/small-ablation/call-center-language-switching/parquet-test-v2 exists; removing contents.
Writing test parquet: 100%|██████████| 1200/1200 [00:04<00:00, 265.68it/s]

Test set: 1200 samples -> ./parquet-test-v2


## Step 8: Verify

In [71]:
ds_train = StreamingDataset(local=MERGED_DIR)
print(f'Train dataset: {len(ds_train)} packed blocks')
sample = ds_train[0]
print(f'  First block: {sample["input_ids"].shape[0]} tokens, {len(sample["attention_mask"])} sequences')

ds_test = StreamingDataset(local=TEST_DIR)
print(f'Test  dataset: {len(ds_test)} samples')

correct = 0
for i in range(len(ds_test)):
    s = ds_test[i]
    meta = json.loads(s['text'])
    last_token = int(s['input_ids'][-1])
    if meta['label'] == 'positive' and last_token == im_end_id:
        correct += 1
    elif meta['label'] == 'negative' and last_token != im_end_id:
        correct += 1
print(f'  Label verification: {correct}/{len(ds_test)} correct ({100 * correct / len(ds_test):.1f}%)')

total_tokens = 0
for i in tqdm(range(len(ds_train)), desc='Counting tokens'):
    total_tokens += ds_train[i]['input_ids'].shape[0]

print(f'\nTrain blocks:  {len(ds_train)}')
print(f'Train tokens:  {total_tokens:,}')
print(f'Train samples: {len(train_samples)} (positive only)')
print(f'Test  samples: {len(test_samples)}')
print(f'Block size:    {BLOCK_SIZE}')

Train dataset: 95496 packed blocks
  First block: 7281 tokens, 4 sequences
Test  dataset: 1200 samples
  Label verification: 1200/1200 correct (100.0%)


Counting tokens: 100%|██████████| 95496/95496 [00:11<00:00, 8146.70it/s]



Train blocks:  95496
Train tokens:  679,544,818
Train samples: 430133 (positive only)
Test  samples: 1200
Block size:    8192


## Step 9: Upload to S3

In [72]:
def s3_sync(local_dir, bucket, prefix):
    s3_path = f's3://{bucket}/{prefix}/'
    print(f'\nUploading {local_dir} -> {s3_path}')
    cmd = ['aws', 's3', 'sync', local_dir, s3_path, '--no-progress']
    subprocess.run(cmd, check=True)
    print(f'Done: {s3_path}')

# s3_sync(MERGED_DIR, S3_BUCKET, f'{S3_PREFIX}/parquet-train-merged')
s3_sync(TEST_DIR, S3_BUCKET, f'{S3_PREFIX}/parquet-test')

print(f'\nReady for training:')
print(f'  --train_file {MERGED_DIR}')
print(f'  --test_file {TEST_DIR}')


Uploading ./parquet-test-v2 -> s3://aies-research-datasets/call-center-language-switching-v2/parquet-test/
upload: parquet-test-v2/index.json to s3://aies-research-datasets/call-center-language-switching-v2/parquet-test/index.json
upload: parquet-test-v2/shard.00000.parquet to s3://aies-research-datasets/call-center-language-switching-v2/parquet-test/shard.00000.parquet
Done: s3://aies-research-datasets/call-center-language-switching-v2/parquet-test/

Ready for training:
  --train_file ./parquet-train-merged-v2
  --test_file ./parquet-test-v2
